In [ ]:
# --- 修复部分开始 ---
# 1. 强制安装 zstd 解压工具 (修复报错的关键)
!sudo apt-get update && sudo apt-get install -y zstd
# --- 修复部分结束 ---

# 2. 安装 Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!pip install aiohttp gradio

import os
import time
import subprocess
import threading

# 3. 在后台启动 Ollama 服务
def start_ollama():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "serve"])

ollama_thread = threading.Thread(target=start_ollama)
ollama_thread.start()

# 等待几秒钟让服务启动
print("正在启动 Ollama 服务，请稍候...")
time.sleep(10)

# 4. 下载模型 (如果之前没下载成功)
print("正在下载 Llama 3.1 模型...")
!ollama run llama3.1 "你好"

# 5. 编写 Chatbot 界面代码
# 5. 编写 Chatbot 界面代码 (已修改为 HWU Student Service Bot)
import gradio as gr
import json
import requests

def chat_response(message, history):
    url = "http://localhost:11434/api/chat"

    # --- 核心修改部分：定义系统提示词 (System Prompt) ---
    # 这里设定了人设 (Role) 和边界 (Constraints)
    system_instruction = """
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).
    Your role is to help students with questions related to:
    - Enrollment and Registration
    - Tuition fees and Finance
    - Campus facilities and Accommodation
    - Student wellbeing and support
    - Visa and Immigration support

    CRITICAL INSTRUCTION:
    If the user asks a question that is NOT related to Heriot-Watt University student services (e.g., coding, general knowledge, jokes, math, other universities), you must STRICTLY refuse to answer.

    In that case, reply EXACTLY with this phrase:
    "I apologize, I can only answer questions regarding Student Services at Heriot-Watt University. Please feel free to ask about that."

    Do not attempt to be helpful with off-topic requests. Stick to this rule strictly.
    """

    # 构建消息列表，第一条必须是 System Prompt
    messages_payload = [{"role": "system", "content": system_instruction}]

    # 追加历史对话 (保持上下文)
    for user_msg, ai_msg in history:
        messages_payload.append({"role": "user", "content": user_msg})
        messages_payload.append({"role": "assistant", "content": ai_msg})

    # 追加当前用户的新问题
    messages_payload.append({"role": "user", "content": message})

    payload = {
        "model": "llama3.1",
        "messages": messages_payload, # 发送包含系统指令的完整列表
        "stream": True
    }

    try:
        response = requests.post(url, json=payload, stream=True)
        response.raise_for_status() # 检查请求是否成功

        partial_message = ""
        for line in response.iter_lines():
            if line:
                decoded_line = line.decode('utf-8')
                json_line = json.loads(decoded_line)
                if 'message' in json_line:
                    content = json_line['message'].get('content', '')
                    partial_message += content
                    yield partial_message
    except Exception as e:
        yield f"Error: {str(e)}"

print("正在启动赫瑞-瓦特大学学生服务助手...")
demo = gr.ChatInterface(
    chat_response,
    title="Heriot-Watt Student Services Bot",
    description="Ask me about enrollment, accommodation, finance, or wellbeing at HWU.",
    examples=["How do I register for my courses?", "Where is the student union?", "I need help with my visa."],
)

demo.launch(share=True, debug=True)

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,341 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:10 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 http://archive.ubuntu.com/ubu

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6daf39811fdf44aa55.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 